# Alpha rhythm in real EEG: PCA artifact removal + FFT (Python)

PhysioNet EEGBCI, subject 1, resting state with **eyes closed** (run 2), 64 channels at 160 Hz.

1. Center the data and compute the $64 \times 64$ spatial covariance matrix.
2. Eigen-decompose it; components with $\lambda > 10 \cdot \mathrm{median}(\lambda)$ are treated as artifacts.
3. Project onto the eigenbasis, zero the artifact components, project back.
4. FFT of the cleaned occipital channel O1 → look for the 8–13 Hz alpha peak.

See [`docs/`](../docs) for the full mathematical derivation.

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import matplotlib.pyplot as plt
from brainwaves.data_loader import load_data, EYES_CLOSED

data, raw = load_data(subject=1, run=EYES_CLOSED)

X = data.T                          # (time samples, channels)
X_centered = X - X.mean(axis=0)
print(f"Data matrix shape: {X_centered.shape}")

## Covariance matrix and its eigenvalues

In [ ]:
covariance_matrix = X_centered.T @ X_centered / (X.shape[0] - 1)
eigenvalues, eigenvectors = np.linalg.eigh(covariance_matrix)   # ascending order

# Heuristic artifact threshold: components with unusually large variance
critical_threshold = 10 * np.median(eigenvalues)
n_good = int(np.sum(eigenvalues <= critical_threshold))
print(f"Artifact components removed: {len(eigenvalues) - n_good} of {len(eigenvalues)}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ids = np.arange(len(eigenvalues))
desc = eigenvalues[::-1]
is_artifact = desc > critical_threshold
ax.bar(ids[is_artifact], desc[is_artifact], color="tab:red", label="removed (artifact)")
ax.bar(ids[~is_artifact], desc[~is_artifact], color="tab:green", label="kept")
ax.axhline(critical_threshold, color="gray", linestyle="--", label="threshold")
ax.set_xlabel("component (sorted by eigenvalue)")
ax.set_ylabel("eigenvalue (variance)")
ax.set_title("Eigenvalues of the spatial covariance matrix")
ax.legend()
plt.show()

## PCA spatial filtering: change of basis → remove artifacts → change back

In [ ]:
data_eig_base = X_centered @ eigenvectors
data_eig_base[:, n_good:] = 0       # zero the largest-variance (artifact) components
X_cleaned = data_eig_base @ eigenvectors.T
print(X_cleaned.shape)

## FFT of the occipital channel O1

In [ ]:
channel_names = raw.info["ch_names"]
fs = raw.info["sfreq"]              # sampling frequency [Hz]
target_data = X_cleaned[:, channel_names.index("O1")]

fft_values = np.fft.rfft(target_data)
fft_frequencies = np.fft.rfftfreq(len(target_data), 1 / fs)
fft_power = np.abs(fft_values) ** 2

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(fft_frequencies, fft_power, color="darkred")
plt.axvspan(8, 13, color="gold", alpha=0.2, label="alpha band (8–13 Hz)")
plt.axvline(10, color="gray", linestyle="--", alpha=0.7, label="10 Hz")
plt.xlim(0, 40)
plt.xlabel("Frequency [Hz]")
plt.ylabel("Power")
plt.title("Power spectrum of channel O1 (eyes closed, after PCA cleaning)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()